# 01 · Dịch dữ liệu benchmark EN → VI

Dịch mỗi văn bản duy nhất (CV, JD) một lần bằng `google/gemini-2.5-flash` (khác model chấm).
Kết quả: `/kaggle/working/translations_vi.jsonl` — mỗi dòng một văn bản, có kiểm tra tỉ lệ độ dài, con số.

**Chạy tiếp được:** nếu hết quota giữa chừng, chạy lại notebook (đính kèm output của lần trước làm input);
văn bản đã dịch ok sẽ được bỏ qua.

In [ ]:
# Tìm dataset đã đính kèm (code ai-service + dữ liệu) trong /kaggle/input
import importlib, json, subprocess, sys
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
engine_file = next(INPUT.rglob("ats_code/app/services/scoring/engine.py"), None)
assert engine_file, "Chưa đính kèm dataset ats-cv-scoring-benchmark (Add Input → Datasets)"
CODE = engine_file.parents[3]                     # .../ats_code
DATA = CODE.parent / "data"
sys.path.insert(0, str(CODE))

if importlib.util.find_spec("pydantic_settings") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic-settings"], check=True)

import kaggle_benchmarks as kbench
from benchmark.common import read_jsonl
from benchmark import kaggle_runner as kr

pairs = read_jsonl(DATA / "pairs.jsonl")
texts_en = {t["hash"]: t for t in read_jsonl(DATA / "texts_en.jsonl")}
print("kaggle_benchmarks", getattr(kbench, "__version__", "?"), "| code:", CODE)
print(len(pairs), "cặp |", len(texts_en), "văn bản duy nhất")

## Cấu hình
`SMOKE_LIMIT = 2` để chạy thử 2 văn bản; `None` để dịch toàn bộ.

In [ ]:
SMOKE_LIMIT = None

In [ ]:
stats = kr.translate_all(
    list(texts_en.values()),
    WORK / "translations_vi.jsonl",
    prior_roots=[INPUT, WORK],
    limit=SMOKE_LIMIT,
)
print(stats)

## Tổng kết

In [ ]:
done = kr.load_translations([INPUT, WORK])
rows = read_jsonl(WORK / "translations_vi.jsonl")
warn = [r for r in rows if r.get("ok") and r.get("issues")]
print(f"Đã có bản dịch: {len(done)}/{len(texts_en)} | lỗi ở lượt này: {sum(not r['ok'] for r in rows)} | "
      f"có cảnh báo: {len(warn)}")